# SIMBA-100 analogues of CEERS nirspec7-000719 / nirspec7-002962

Find simulated analogues (redshift, stellar mass, mass-weighted age) of the two observed quiescent
galaxies from `pilot_specphot`, split them into AGN / non-AGN hosts, and compare:

1. their **powderday SEDs** (median + 16-84% band, dust-on) against the observed **CIGALE best-fit
   SED** (`cigale_ppxfprior_agn`, the adopted fiducial run) and the observed photometry;
2. their **simulated dust masses** against the CIGALE Bayesian dust-mass estimates.

**This notebook now reads the CORRECTED (Chabrier) run set** staged by
`analogues_specphot_cis100_rt_agn.ipynb` (2026-07-30): selection re-centred on the CIGALE
Chabrier masses, AGN-off SEDs assembled per galaxy from two physically identical trees
(`agnoff_source_map_chab.fits`: 59 reused from `dusty_simdust`, 57 from
`dusty_simdust_chab_topup`), AGN-on in `dusty_simdust_chab_agn`, dust-free arm in
`nodust_chab_agn`. The analogues sit at essentially the observed redshifts:

| target | z_obs | snap | z_snap | \|dz\| | AGN | non-AGN | N |
|---|---|---|---|---|---|---|---|
| 000719 | 1.463 | 091 | 1.4584 | 0.005 | 20 | 40 | 60 |
| 002962 | 1.266 | 096 | 1.2780 | 0.012 | 16 | 40 | 56 |

**What changed vs the previous version.** It read `sed_dust_comparison/dusty_simdust`, whose RT only
ever completed at snap >= 100 (z <= 1.145), together with the 636-galaxy
`forward_modeled_unique_sample.fits` parent list, and then matched with loose cosmic-time windows.
That forced every analogue *below* both target redshifts and is the origin of the redshift mismatch
in the earlier figures - which were rendered on 2026-07-28 ~11:00, before this RT finished at 16:38
and 03:27. Now:

- AGN-off SEDs -> per galaxy via `agnoff_source_map_chab.fits` (`dusty_simdust` or
  `dusty_simdust_chab_topup`); `rtout_path()` resolves the tree
- sample -> `analogues_specphot/rt_selection_chab_{719,2962}.fits`
- Parts 3-4 collapse to a table read: no BH harvest, no time-window matching
- **Part 5 is plotted in the REST FRAME** (see below)

**Rest frame.** powderday SEDs are natively rest-frame nu*Lnu [erg/s], so they are used untouched
and the observed CIGALE model + photometry are de-redshifted into rest-frame luminosity via
nu_obs*F_nu*4*pi*d_L^2 = nu_rest*L_nu,rest. With |dz| <= 0.012 the frame choice now shifts things by
< 1%, so any residual offset is physical rather than a redshift artefact.

**AGN: two definitions, both reported.** `IS_AGN = (f_Edd >= 0.02) & (L_bol >= 1e43)` is the staged
column and remains the primary split. `IS_AGN_LBOL = (L_bol >= 1e43)` is added alongside because the
f_Edd gate is what zeroes out the AGN class in the cis25 box (max f_Edd there ~2e-3 while L_bol
reaches 1.1e44, i.e. above CIGALE's L_AGN ~ 3.8e43 for 002962). Part 4 prints the cross-tab; flip
`SPLIT_BY` in Part 5 to plot either.

**Caveat that still applies: no AGN emission in THIS run.** `parameters_master.py` here has
`BH_SED = False`, so the simulated SEDs contain no torus even for AGN hosts, and the split tests the
*host*. For 002962 compare the AGN-host band to the CIGALE **galaxy-only** curve (dashed), never to
the total. The AGN-on twin is staged by `analogues_specphot_cis100_rt_agn.ipynb`
(`sed_analogues/dusty_simdust_chab_agn/`, `BH_SED=True` — the old pointer named
`dusty_simdust_agn`, a tag that never existed on disk, which is why `AGN_SED_ROOT` silently
stayed `None`). Part 0 now points at the correct tag and Part 5 overlays every galaxy whose
AGN-on RT is finished, printing the per-band AGN-on/AGN-off ratio; `NODUST_SED_ROOT` points at
the dust-free arm.



**Observed targets** (fiducial values from `final_consensus.md` / `cigale_ppxfprior_agn`):

| | z_spec | log M*/Msun | age_mw [Gyr] | log sSFR | AGN | log Mdust/Msun |
|---|---|---|---|---|---|---|
| 000719 | 1.463 | 10.54 +- 0.05 (bimodal branch at 9.98) | ~2.4 (branch at 0.9) | -10.5..-10.9 | no (fracAGN best = 0) | 6.74 (err > value -> ~upper limit) |
| 002962 | 1.266 | 10.92 +- 0.05 | ~3.1-3.5 | -11.4 | yes, obscured type-2 (fracAGN ~ 0.5) | 7.17 +- ~0.5 |

**Convergence.** Every RT output carries `converged = 'no'` (stopped at the 7-iteration cap set by `set_n_initial_iterations(7)` in powderday's `front_end_tools.py`). Checked against the run logs: the convergence metric is **flat** at 1.07-1.24 vs the 1.01 threshold with `Difference from previous iteration: 1.00` (zero improvement) from iteration 2 onward, and each log has 45 `specific_energy below minimum allowed` warnings. This is a Monte-Carlo **photon** noise floor in the sparsest outskirt cells of the +/-100 kpc box, not an iteration shortfall - raising the cap would change nothing. Since the criterion is the worst 1% of cells (negligible dust mass, negligible SED contribution), it is largely cosmetic. The lever, if ever needed, is `n_photons_initial` (currently 1e7), not more iterations.


In [ ]:
# ── Part 0 · config ──────────────────────────────────────────────────────────
import os, warnings
import numpy as np
import h5py
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.table import Table
from astropy.cosmology import Planck13          # same cosmology as simbanator.sed.makesed
import astropy.units as u
from astropy import constants as const

from simbanator.io.simba import Simulation

sim = Simulation('cis100')
plt.rcParams.update({'font.size': 13})

HOME        = '/mnt/home/glorenzon/analize_simba_cgm'
CIS_OUT     = os.path.join(HOME, 'output', 'cis100')

# ── CORRECTED (Chabrier) run set, staged by analogues_specphot_cis100_rt_agn.ipynb ──
# AGN-off: each corrected-window galaxy's SED lives in ONE of two physically identical trees
# (59 finished in dusty_simdust, 57 in dusty_simdust_chab_topup); agnoff_source_map_chab.fits
# records which, and rtout_path()/read_powderday_sed() resolve it per galaxy (root=None).
SED_BASE    = os.path.join(CIS_OUT, 'sed_analogues')
SED_ROOT    = os.path.join(SED_BASE, 'dusty_simdust', 'powderday_sed_out')  # reuse tree (59 gals)

# AGN-on twin (BH_SED=True, Hopkins) and dust-free arm. The old pointer said
# 'dusty_simdust_agn' — a tag that never existed on disk — so AGN_SED_ROOT silently stayed
# None and Part 5 never overlaid the AGN. Staged directories exist BEFORE the RT finishes,
# so completeness is checked per galaxy (readable Peeled/seds, Part 2), never by isdir/exists.
AGN_SED_ROOT    = os.path.join(SED_BASE, 'dusty_simdust_chab_agn', 'powderday_sed_out')
NODUST_SED_ROOT = os.path.join(SED_BASE, 'nodust_chab_agn', 'powderday_sed_out')
if not os.path.isdir(AGN_SED_ROOT):
    AGN_SED_ROOT = None
NENKOVA_SED_ROOT = os.path.join(SED_BASE, 'dusty_simdust_chab_agn_nenkova', 'powderday_sed_out')
if not os.path.isdir(NENKOVA_SED_ROOT):
    NENKOVA_SED_ROOT = None
if not os.path.isdir(NODUST_SED_ROOT):
    NODUST_SED_ROOT = None

OBS_DIR     = os.path.join(HOME, 'obs_data', 'pilot_specphot_cigale_chab')   # staged CIGALE fiducial outputs
OUTDIR      = os.path.join(CIS_OUT, 'analogues_specphot')
# per-target selection tables from the staging notebook: they already carry every property used
# here, so Parts 3-4 are just a table read. chab = re-selected on the CIGALE Chabrier masses;
# the rt_selection_{tid}.fits (no 'chab') are the Salpeter-era tables of the old selection.
SEL_FITS    = os.path.join(OUTDIR, 'rt_selection_chab_{tid}.fits')
os.makedirs(OUTDIR, exist_ok=True)

# (snap, gid) -> powderday_sed_out root holding that galaxy's AGN-off SED
AGNOFF_MAP = {}
for _r in Table.read(os.path.join(OUTDIR, 'agnoff_source_map_chab.fits')):
    _tag = _r['AGNOFF_TAG']
    _tag = (_tag.decode() if isinstance(_tag, (bytes, np.bytes_)) else str(_tag)).strip()
    AGNOFF_MAP[(int(_r['SNAPSHOT']), int(_r['GROUPID_SNAPSHOT']))] = \
        os.path.join(SED_BASE, _tag, 'powderday_sed_out')

# fixed entity colors (Okabe-Ito CVD-safe pair); observed data is always black
COL = {'AGN': '#E69F00', 'no_AGN': '#0072B2', 'obs': 'k'}

# ── observed targets (fiducial: cigale_ppxfprior_agn + final_consensus.md) ──
# Selection windows are NOT applied here any more - they were applied at staging time, directly on
# the full m100n1024 population at the matching snapshot. Recorded for reference (Chabrier):
#   719  -> snap 091 (z=1.4584), 10.29+-0.30 dex, 2.65+-0.85 Gyr -> 60 staged (20 AGN / 40 non-AGN)
#   2962 -> snap 096 (z=1.2780), 10.68+-0.40 dex, 3.30+-1.00 Gyr -> 56 staged (16 AGN / 40 non-AGN)
TARGETS = {
    '719': dict(
        z=1.463, snap=91, z_snap=1.4584,
        # age 2.65 Gyr = mass-weighted age MC-propagated from the Chabrier CIGALE bayes SFH —
        # the value the chab selection window was centred on (2.35 was the Salpeter-run age)
        logm=10.29, logm_err=0.05, age_gyr=2.65, agn=False,
        logmdust=6.74, logmdust_err=0.6,          # err > value -> treat as ~upper limit
        best_model='719_best_model.fits', obs_id='719',
        # known age/mass bimodality: the Bagpipes-preferred young branch (0.9 Gyr, logM*=9.98)
        # lies below the staged mass window -> not matchable here.
    ),
    '2962': dict(
        z=1.266, snap=96, z_snap=1.2780,
        logm=10.68, logm_err=0.05, age_gyr=3.30, agn=True,
        logmdust=7.18, logmdust_err=0.5,
        best_model='2962_best_model.fits', obs_id='2962',
    ),
}

# ── AGN host classification (applied at staging time, at the analogue's own snapshot) ──
# 002962 is a *radiatively* luminous obscured AGN (L_AGN ~ 3.8e43 erg/s from CIGALE), so the
# bolometric criterion is the physically motivated one; the f_Edd gate is reported alongside
# because it is what removes the entire AGN class in the cis25 box.
# NOTE: this run has BH_SED=False -> no torus in the simulated SEDs; the split tests the HOST.
# ── IMF consistency ──────────────────────────────────────────────────────────
# CIGALE has been RE-RUN with bc03 `imf = 1` = CHABRIER (cigale_ppxfprior_agn_chab), matching
# powderday/FSPS `imf_type = 1`. The observed masses below are therefore native Chabrier and no
# conversion is applied. Measured Salpeter->Chabrier shift from the two runs: -0.255 dex (719),
# -0.241 dex (2962); chi2_red and fracAGN were unchanged, and M_dust is IMF-independent.
# The old Salpeter run (obs_data/pilot_specphot_cigale) over-boosted the mass-rescaled simulated
# SEDs by ~1.7x and was the reason the simulated continuum sat ~2x above the photometry.
IMF_SALP_TO_CHAB_DEX = 0.0        # already Chabrier; kept so Part 5 needs no edit
for _T in TARGETS.values():
    _T['logm_chab'] = _T['logm'] - IMF_SALP_TO_CHAB_DEX
print(f"IMF: CIGALE re-run with Chabrier (imf=1); masses native "
      f"(719 {TARGETS['719']['logm']:.2f}, 2962 {TARGETS['2962']['logm']:.2f})")

EPS_RAD      = 0.1
AGN_LBOL_MIN = 1e43        # erg/s
AGN_FEDD_MIN = 0.02

print('AGN-off SEDs ->', f'{len(AGNOFF_MAP)} galaxies mapped across dusty_simdust + dusty_simdust_chab_topup')
print('AGN-on tree  ->', AGN_SED_ROOT or 'not staged yet (run analogues_specphot_cis100_rt_agn.ipynb)')
print('dust-free    ->', NODUST_SED_ROOT or 'not staged yet')
print('Nenkova tree ->', NENKOVA_SED_ROOT or 'not staged yet')
print('output       ->', OUTDIR)

In [ ]:
# ── Part 1 · observed CIGALE SEDs + photometry ───────────────────────────────
# best_model.fits: wavelength [nm, observed frame], Fnu [mJy], plus L_lambda component columns.
# Galaxy-only curve: rescale Fnu by the luminosity fraction NOT in the SKIRTOR AGN components
# (ratio is unitless, so the L_lambda units don't matter).
AGN_COLS = ('agn.SKIRTOR2016_torus', 'agn.SKIRTOR2016_polar_dust', 'agn.SKIRTOR2016_disk')

def load_best_model(fname):
    with fits.open(os.path.join(OBS_DIR, fname)) as f:
        d = f[1].data
        wl_um  = np.asarray(d['wavelength'], float) * 1e-3        # nm -> micron (observed frame)
        fnu    = np.asarray(d['Fnu'], float)                      # mJy
        ltot   = np.asarray(d['L_lambda_total'], float)
        lagn   = np.zeros_like(ltot)
        for c in AGN_COLS:
            if c in d.columns.names:
                lagn += np.asarray(d[c], float)
        with np.errstate(invalid='ignore', divide='ignore'):
            fgal = fnu * np.clip(1.0 - lagn / ltot, 0.0, 1.0)
    s = np.argsort(wl_um)
    return wl_um[s], fnu[s], fgal[s]

# pivot wavelengths [micron] of the fitted bands (columns of observations.fits)
OBS_BANDS = {
    'cfht.megacam.u': 0.381, 'cfht.megacam.g': 0.487, 'cfht.megacam.r': 0.625,
    'cfht.megacam.i': 0.770, 'cfht.megacam.z': 0.888,
    'hst.acs.wfc.F606W': 0.592, 'hst.acs.wfc.F814W': 0.805,
    'hst.wfc3.ir.F125W': 1.248, 'hst.wfc3.ir.F160W': 1.537,
    'cfht.wircam.J': 1.253, 'cfht.wircam.H': 1.631, 'cfht.wircam.Ks': 2.146,
    'spitzer.irac.I1': 3.55, 'spitzer.irac.I2': 4.49, 'spitzer.irac.I3': 5.73, 'spitzer.irac.I4': 7.87,
    'jwst.miri.F770W': 7.70, 'jwst.miri.F1000W': 10.0, 'jwst.miri.F1280W': 12.8,
    'jwst.miri.F1500W': 15.0, 'jwst.miri.F1800W': 18.0, 'jwst.miri.F2100W': 21.0,
}

def load_observations(obs_id):
    """(wl_um, flux_mJy, err_mJy) for the fitted bands of one galaxy; NaN/absent bands dropped."""
    with fits.open(os.path.join(OBS_DIR, 'observations.fits')) as f:
        d = f[1].data
        row = d[np.asarray([str(i).strip() for i in d['id']]) == obs_id][0]
    wl, fl, er = [], [], []
    for band, piv in OBS_BANDS.items():
        if band not in d.columns.names:
            continue
        v = float(row[band]); e = float(row[band + '_err'])
        if np.isfinite(v) and v > 0:
            wl.append(piv); fl.append(v); er.append(e)
    s = np.argsort(wl)
    return np.asarray(wl)[s], np.asarray(fl)[s], np.asarray(er)[s]

OBS = {}
for tid, T in TARGETS.items():
    wl_m, f_tot, f_gal = load_best_model(T['best_model'])
    wl_p, f_p, e_p = load_observations(T['obs_id'])
    OBS[tid] = dict(model_wl=wl_m, model_tot=f_tot, model_gal=f_gal, phot_wl=wl_p, phot_f=f_p, phot_e=e_p)
    print(f"{tid}: best-model {wl_m.size} pts, photometry {wl_p.size} bands "
          f"({wl_p.min():.2f}-{wl_p.max():.1f} um)")


In [ ]:
# ── Part 2 · RT completeness on the correct-epoch tree ───────────────────────
def rtout_path(snap, gid, root=None):
    """root=None -> this galaxy's AGN-off tree (dusty_simdust or chab_topup, per the map)."""
    if root is None:
        root = AGNOFF_MAP.get((int(snap), int(gid)), SED_ROOT)
    return os.path.join(root, f'snap_{int(snap):03d}', f'gal_{int(gid)}',
                        f'snap{int(snap):03d}.galaxy{int(gid):06d}.rtout.sed')

def sed_is_complete(path):
    """True only if the RT FINISHED. hyperion creates .rtout.sed when it STARTS and writes
    the Peeled SED groups last, so os.path.exists says yes hours before the file is readable
    (see the Part 6g module docstring for the 2026-07-31 case)."""
    if not os.path.exists(path):
        return False
    try:
        with h5py.File(path, 'r') as f:
            g = f.get('Peeled')
            return bool(g) and any('seds' in g[k] for k in g.keys())
    except OSError:
        return False

# The old parent-sample scan is gone: the staging notebook selected directly from the full
# m100n1024 population at the matching snapshot, so the selection tables ARE the sample.
# Here we only verify that every staged galaxy has a finished RT.
print(f"{'target':>7} {'snap':>5} {'z_snap':>8} {'N':>4} {'SED':>5} {'AGN-on SED':>11}")
for tid, T in TARGETS.items():
    S = Table.read(SEL_FITS.format(tid=tid))
    has = np.array([sed_is_complete(rtout_path(s, g))
                    for s, g in zip(S['SNAPSHOT'], S['GROUPID_SNAPSHOT'])])
    if AGN_SED_ROOT:
        hasa = np.array([sed_is_complete(rtout_path(s, g, AGN_SED_ROOT))
                         for s, g in zip(S['SNAPSHOT'], S['GROUPID_SNAPSHOT'])])
        na = f'{int(hasa.sum())}/{len(S)}'
    else:
        na = '-'
    print(f"{tid:>7} {int(T['snap']):5d} {T['z_snap']:8.4f} {len(S):4d} "
          f"{int(has.sum()):5d} {na:>11}")
    if not has.all():
        print('   !! missing RT for:',
              [f"{int(s)}/{int(g)}" for s, g, h in
               zip(S['SNAPSHOT'], S['GROUPID_SNAPSHOT'], has) if not h])

In [ ]:
# ── Part 3 · BH mass / accretion  ────────────────────────────────────────────
# NOT NEEDED ANY MORE. The staging notebook harvested FEDD / LBOL / IS_AGN straight from the
# caesar catalog at the matching snapshot into rt_selection_{tid}.fits, so the old per-snapshot
# BH harvest (and its bh_at_snapshot.fits cache) is obsolete. Kept as a no-op so the Part
# numbering still matches the write-up.
print('Part 3 skipped: BH quantities already in rt_selection_chab_{719,2962}.fits '
      '(columns FEDD, LBOL, IS_AGN).')

In [ ]:
# ── Part 4 · analogue sample (direct table read) ─────────────────────────────
# Every galaxy already sits at the correct epoch, mass and age and carries AGN_CLASS, so the old
# funnel (cosmic-time window + mass/age windows + HAS_SED) is unnecessary - re-applying it here is
# exactly what used to re-introduce the redshift spread.
def _as_str(col):
    """FITS string columns come back as bytes (dtype |S6). Comparing those to 'AGN' silently
    matches NOTHING, which would empty every AGN/non-AGN split downstream (Parts 4b/5/6).
    The old version never hit this because it built AGN_CLASS in memory with np.where."""
    return np.array([x.decode() if isinstance(x, (bytes, np.bytes_)) else str(x) for x in col])

ANALOGUES = {}
for tid, T in TARGETS.items():
    A = Table.read(SEL_FITS.format(tid=tid))
    if 'AGN_CLASS' not in A.colnames:                     # defensive: rebuild from IS_AGN
        A['AGN_CLASS'] = np.where(np.asarray(A['IS_AGN'], bool), 'AGN', 'no_AGN')
    A['AGN_CLASS'] = _as_str(A['AGN_CLASS'])              # bytes -> str (see above)

    # second, L_bol-only definition (no Eddington-ratio gate) - reported alongside
    lbol = np.asarray(A['LBOL'], float)
    fedd = np.asarray(A['FEDD'], float)
    A['IS_AGN_LBOL']    = np.nan_to_num(lbol) >= AGN_LBOL_MIN
    A['AGN_CLASS_LBOL'] = np.where(np.asarray(A['IS_AGN_LBOL']), 'AGN', 'no_AGN')
    A.sort(['SNAPSHOT', 'GROUPID_SNAPSHOT'])
    ANALOGUES[tid] = A

    agn_m  = np.asarray(A['AGN_CLASS']) == 'AGN'
    agn_lb = np.asarray(A['IS_AGN_LBOL'], bool)
    t_obs = Planck13.age(T['z']).to(u.Gyr).value
    t_sim = Planck13.age(T['z_snap']).to(u.Gyr).value
    print(f"── {tid}  (z_obs={T['z']}, snap {int(T['snap'])} at z={T['z_snap']}) ──")
    print(f"  N = {len(A)}   |dz| = {abs(T['z'] - T['z_snap']):.4f}   "
          f"|dt| = {abs(t_obs - t_sim) * 1e3:.0f} Myr")
    print(f"  AGN (f_Edd>={AGN_FEDD_MIN} & L_bol>={AGN_LBOL_MIN:.0e}) : "
          f"{int(agn_m.sum()):3d} | non-AGN {int((~agn_m).sum()):3d}")
    print(f"  AGN (L_bol>={AGN_LBOL_MIN:.0e} only)              : "
          f"{int(agn_lb.sum()):3d} | non-AGN {int((~agn_lb).sum()):3d}")
    print(f"    cross-tab  both {int((agn_m & agn_lb).sum()):3d} | "
          f"L_bol-only {int((~agn_m & agn_lb).sum()):3d} | neither {int((~agn_m & ~agn_lb).sum()):3d}")
    print(f"    f_Edd  max {np.nanmax(fedd):.3e}   L_bol max {np.nanmax(lbol):.3e} erg/s")
    print(f"  log M*  : obs {T['logm']:.2f}  | analogues {np.median(A['LOG_MSTAR']):.2f} "
          f"[{np.percentile(A['LOG_MSTAR'], 16):.2f}, {np.percentile(A['LOG_MSTAR'], 84):.2f}]")
    print(f"  age     : obs {T['age_gyr']:.2f}  | analogues {np.median(A['AGE_STAR']):.2f} "
          f"[{np.percentile(A['AGE_STAR'], 16):.2f}, {np.percentile(A['AGE_STAR'], 84):.2f}] Gyr")
    out = os.path.join(OUTDIR, f'analogues_{tid}.fits')
    A.write(out, overwrite=True)
    print(f"  -> {out}\n")

In [ ]:
# ── Part 4b · sanity check: are the analogues really analogous? ──────────────
# Distributions of the matching properties for the SELECTED analogues vs the observed values
# (consensus ranges shaded). With loose windows the selection can sit asymmetrically inside the
# window (the sample has few galaxies as old/massive as the targets and no SEDs above z=1.145),
# and the SED normalization inherits that bias: younger / more massive analogues -> brighter
# stellar continuum. Check the printed medians before interpreting the SED comparison; consider
# RESCALE_TO_TARGET_MASS in Part 5 to factor the mass offset out.
# logm ranges in the CHABRIER system, consistent with the chab selection and TARGETS (CIGALE
# chab re-run: 719 10.29, 2962 10.68; measured Salpeter->Chab shifts -0.255 / -0.241 dex).
# Bagpipes masses (Kroupa ~ Chabrier) are kept as published.
OBS_RANGES = {
    '719':  dict(logm=(10.24, 10.34), logm_alt=9.98,   # fiducial +/- err; alt = Bagpipes young branch
                 age=(2.1, 3.0),      age_alt=0.9),    # CIGALE mw .. pPXF 2.97; alt = PSB branch
    '2962': dict(logm=(10.63, 10.76), logm_alt=None,   # CIGALE-chab 10.68-err .. Bagpipes 10.76
                 age=(3.1, 3.6),      age_alt=None),
}

fig, axes = plt.subplots(len(TARGETS), 3, figsize=(13, 4 * len(TARGETS)))
for irow, (tid, T) in enumerate(TARGETS.items()):
    A, R = ANALOGUES[tid], OBS_RANGES[tid]
    agn_m = np.asarray(A['AGN_CLASS']) == 'AGN'
    panels = [('LOG_MSTAR', r'$\log\,M_*\,[M_\odot]$', T['logm'],    R['logm'], R['logm_alt']),
              ('AGE_STAR',  'mass-weighted age [Gyr]', T['age_gyr'], R['age'],  R['age_alt']),
              ('REDSHIFT',  'redshift',                T['z'],       None,      None)]
    for icol, (key, xlabel, obs_val, obs_rng, alt) in enumerate(panels):
        ax = axes[irow, icol]
        x = np.asarray(A[key], float)
        lo = min(x.min(), alt if alt is not None else obs_val, obs_val)
        hi = max(x.max(), obs_val)
        pad = 0.06 * (hi - lo)
        bins = np.linspace(lo - pad, hi + pad, 25)
        ax.hist(x, bins=bins, color='0.85', label='all analogues')
        ax.hist(x[~agn_m], bins=bins, histtype='step', lw=1.8, color=COL['no_AGN'], label='non-AGN')
        ax.hist(x[agn_m],  bins=bins, histtype='step', lw=1.8, color=COL['AGN'],    label='AGN')
        if obs_rng is not None:
            ax.axvspan(obs_rng[0], obs_rng[1], color='k', alpha=0.12, lw=0, label='obs. consensus')
        ax.axvline(obs_val, color='k', lw=1.6, label='observed')
        if alt is not None:
            ax.axvline(alt, color='k', lw=1.4, ls=':', label='alt. branch')
        med = float(np.median(x))
        ax.axvline(med, color='crimson', lw=1.4, ls='--', label='analogue median')
        ax.set_xlabel(xlabel)
        if icol == 0:
            ax.set_ylabel(f'nirspec7-{tid.zfill(6)}\nN analogues')
        if irow == 0 and icol == 2:
            ax.legend(fontsize=9, frameon=False)
        m_agn = float(np.median(x[agn_m])) if agn_m.any() else np.nan
        m_non = float(np.median(x[~agn_m])) if (~agn_m).any() else np.nan
        print(f'{tid} {key:9s}: obs {obs_val:6.2f} | analogue median {med:6.2f} '
              f'(AGN {m_agn:6.2f} / non-AGN {m_non:6.2f})')
    print()
fig.tight_layout()
for ext in ('png', 'pdf'):
    fig.savefig(os.path.join(OUTDIR, f'analogue_property_sanity.{ext}'), dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# ── Part 5 · powderday SED stacks vs observed CIGALE SED — OBSERVED-FRAME Fnu ─
from hyperion.model import ModelOutput

MAX_INDIV = 40      # max individual (thin) curves drawn per group; stack always uses ALL
SPLIT_BY  = 'AGN_CLASS'        # or 'AGN_CLASS_LBOL' to split on the L_bol-only definition

# OBSERVED-FRAME comparison (changed 2026-08-03; the previous version worked in rest-frame
# nu*Lnu and converted the observations instead). powderday SEDs are natively rest-frame
# nu*Lnu [erg/s]; each mock is placed at its TARGET's redshift (Planck13):
#     Fnu_obs = nuLnu_rest / (4 pi dL^2 nu_obs),   lambda_obs = lambda_rest (1+z)
# so the CIGALE best model and the photometry are plotted untouched (mJy). Stacking still
# happens in rest-frame nu*Lnu on GRID — the conversion is a fixed per-wavelength factor
# applied at plot time, so medians/percentiles are unaffected by the change of variable.
_SED_CACHE = {}
def read_powderday_sed(snap, gid, root=None):
    """(wav_rest_um, nuLnu_erg_s) from one .rtout.sed (largest aperture, mean over sightlines).

    root=None -> this galaxy's AGN-off tree, resolved per galaxy by rtout_path/AGNOFF_MAP."""
    key = (int(snap), int(gid), root or 'agnoff')
    if key not in _SED_CACHE:
        m = ModelOutput(rtout_path(snap, gid, root))
        sed = m.get_sed(inclination='all', aperture=-1)   # -> (n_incl, n_wav), units erg/s
        wav = np.asarray(sed.wav, float)                  # micron, rest frame
        val = np.atleast_2d(np.asarray(sed.val, float))
        # THETA=[0,45,90,135] / PHI=[0,90,180,270] are 4 genuine sightlines, but for these
        # dust-poor quenched galaxies they agree to <0.5%, so the mean is safe (and the lack of
        # inclination scatter is a result, not an assumption).
        val = val.mean(axis=0)
        s = np.argsort(wav)
        _SED_CACHE[key] = (wav[s], val[s])
    return _SED_CACHE[key]

def obs_to_rest_nuLnu(wl_obs_um, fnu_mJy, z):
    """Observed-frame (wl [um], Fnu [mJy]) -> rest-frame (wl [um], nu*Lnu [erg/s]).

    Kept for reference/other cells; the plots below no longer convert the observations.
    """
    wl = np.asarray(wl_obs_um, float)
    dl = Planck13.luminosity_distance(z).to(u.cm)
    nu = (const.c / (wl * u.micron)).to(u.Hz)
    fnu = (np.asarray(fnu_mJy, float) * u.mJy).to(u.erg / u.s / u.cm**2 / u.Hz)
    return wl / (1.0 + z), (nu * fnu * 4 * np.pi * dl**2).to(u.erg / u.s).value

GRID = np.geomspace(0.1, 500.0, 300)     # REST-frame micron (stacking grid)

def obs_fnu_factor(z):
    """(lambda_obs [um], factor) turning nu*Lnu(GRID) [erg/s] into observed Fnu [mJy] at z."""
    wav_obs = GRID * (1.0 + z)
    dl = Planck13.luminosity_distance(z).to(u.cm).value
    nu_obs = (const.c / (wav_obs * u.micron)).to(u.Hz).value
    return wav_obs, 1.0 / (4.0 * np.pi * dl**2 * nu_obs) / 1e-26   # 1 mJy = 1e-26 cgs

def stack(A_group, tag='', logm_target=None, root=None):
    """Median and 16/84 percentile nu*Lnu(GRID) over one analogue group; also per-galaxy curves."""
    curves = []
    for i, r in enumerate(A_group):
        try:
            wl, nuLnu = read_powderday_sed(r['SNAPSHOT'], r['GROUPID_SNAPSHOT'], root)
        except (OSError, IOError):
            continue                                    # missing/unfinished RT -> skip galaxy
        if logm_target is not None:
            nuLnu = nuLnu * 10.0 ** (logm_target - float(r['LOG_MSTAR']))
        with np.errstate(invalid='ignore', divide='ignore'):
            lg = np.interp(np.log10(GRID), np.log10(wl), np.log10(nuLnu),
                           left=np.nan, right=np.nan)
        curves.append(10**lg)
        if (i + 1) % 25 == 0:
            print(f'  [{tag}] {i + 1}/{len(A_group)} SEDs read')
    C = np.asarray(curves)
    if C.size == 0:
        return None
    return dict(med=np.nanmedian(C, axis=0), lo=np.nanpercentile(C, 16, axis=0),
                hi=np.nanpercentile(C, 84, axis=0), curves=C, n=len(C))

# AGN-on trees: Hopkins (AGN_SED_ROOT, Part 0) + Nenkova CLUMPY i=90 (staged 2026-08-03;
# fallback here in case Part 0 predates it). Both are the SAME galaxies as the AGN-off run —
# only the injected BH template differs — so solid vs dotted vs dash-dotted IS the torus.
if 'NENKOVA_SED_ROOT' not in globals():
    NENKOVA_SED_ROOT = os.path.join(SED_BASE, 'dusty_simdust_chab_agn_nenkova',
                                    'powderday_sed_out')
    if not os.path.isdir(NENKOVA_SED_ROOT):
        NENKOVA_SED_ROOT = None

# Mass rescaling multiplies each analogue by M*_target / M*_analogue. With correct-epoch selection
# the mass offset is already small, so the rescaled and un-rescaled panels should nearly agree -
# which is itself the check that the new selection removed the old bias. Both are produced.
for RESCALE_TO_TARGET_MASS in (False, True):
    rng_sed = np.random.default_rng(7)
    for tid, T in TARGETS.items():
        A = ANALOGUES[tid]
        # Chabrier mass: the simulation is Chabrier, CIGALE's M* is Salpeter (see Part 0)
        _logm_t = T['logm_chab'] if RESCALE_TO_TARGET_MASS else None
        z = T['z']
        WOBS, FAC = obs_fnu_factor(z)
        fig, ax = plt.subplots(figsize=(9, 6.5))
        for cls in ('no_AGN', 'AGN'):
            G = A[np.asarray(A[SPLIT_BY]) == cls]
            if len(G) == 0:
                continue
            S = stack(G, tag=f'{tid}/{cls}', logm_target=_logm_t)
            if S is None:
                continue
            c = COL[cls]
            idx = np.arange(len(S['curves']))
            if idx.size > MAX_INDIV:
                idx = rng_sed.choice(idx, MAX_INDIV, replace=False)
            for cv in S['curves'][idx]:
                ax.plot(WOBS, cv * FAC, color=c, lw=0.6, alpha=0.15)
            ax.fill_between(WOBS, S['lo'] * FAC, S['hi'] * FAC, color=c, alpha=0.28, lw=0)
            lab = 'AGN host' if cls == 'AGN' else 'non-AGN host'
            ax.plot(WOBS, S['med'] * FAC, color=c, lw=2, label=f'SIMBA {lab} (N={S["n"]})')
            # AGN-on twins (BH_SED=True): same galaxies, only the injected template differs
            if cls == 'AGN':
                for root, ls, rlab in ((AGN_SED_ROOT, ':', 'Hopkins'),
                                       (NENKOVA_SED_ROOT, '-.', 'Nenkova i=90')):
                    if not root:
                        continue
                    SA = stack(G, tag=f'{tid}/{cls}/{rlab}', logm_target=_logm_t, root=root)
                    if SA is None:
                        continue
                    ax.plot(WOBS, SA['med'] * FAC, color=c, lw=2, ls=ls,
                            label=f'SIMBA {lab}, AGN-on {rlab} (N={SA["n"]})')
                    with np.errstate(invalid='ignore', divide='ignore'):
                        ratio = SA['med'] / S['med']
                    for wl_probe in (1.0, 5.0, 10.0, 20.0, 100.0):
                        j = int(np.argmin(np.abs(GRID - wl_probe)))
                        print(f'  [{tid}] {rlab}/AGN-off at {wl_probe:6.1f} um rest: '
                              f'{ratio[j]:.3f}')

        O = OBS[tid]
        ax.plot(O['model_wl'], O['model_tot'], color=COL['obs'], lw=1.6,
                label='CIGALE best model (total)')
        if T['agn']:
            ax.plot(O['model_wl'], O['model_gal'], color=COL['obs'], lw=1.4, ls='--',
                    label='CIGALE galaxy only (AGN subtracted)')
        ax.errorbar(O['phot_wl'], O['phot_f'], yerr=np.abs(O['phot_e']), fmt='o', ms=6,
                    mfc='w', mec=COL['obs'], ecolor=COL['obs'], elinewidth=1, capsize=2,
                    zorder=5, label='observed photometry')

        ax.set(xscale='log', yscale='log', xlim=(0.2, 1000),
               xlabel=r'observed-frame wavelength [$\mu$m]',
               ylabel=r'$F_\nu$ [mJy]')
        fin = np.asarray(O['phot_f'], float)
        fin = fin[np.isfinite(fin) & (fin > 0)]
        ax.set_ylim(fin.min() / 3e2, fin.max() * 3e2)
        # same axis relabelled in the rest frame (NOT a second scale)
        axt = ax.secondary_xaxis('top', functions=(lambda x, _z=z: x / (1 + _z),
                                                   lambda x, _z=z: x * (1 + _z)))
        axt.set_xlabel(r'rest-frame wavelength [$\mu$m]', fontsize=10)
        ax.grid(alpha=0.2, which='both', lw=0.5)
        ax.legend(loc='lower center', fontsize=9, frameon=False)
        _rs = ', mass-rescaled' if RESCALE_TO_TARGET_MASS else ''
        ax.set_title(f'nirspec7-{tid.zfill(6)}  (z={z})  vs  SIMBA-100 analogues at snap '
                     f'{int(T["snap"])} (z={T["z_snap"]})\nobserved-frame $F_\\nu$, '
                     f'powderday dust-on{_rs}', fontsize=12)
        fig.tight_layout()
        _suffix = '_massrescaled' if RESCALE_TO_TARGET_MASS else ''
        for ext in ('png', 'pdf'):
            fig.savefig(os.path.join(OUTDIR, f'sed_analogues_{tid}_obsfnu{_suffix}.{ext}'),
                        dpi=200, bbox_inches='tight')
        plt.show()
print(f'split used: {SPLIT_BY}')
if AGN_SED_ROOT is None:
    print('note: this run has BH_SED=False -> no AGN emission. For 2962 compare the AGN-host band '
          'to the DASHED (galaxy-only) observed curve in the mid-IR.')


In [ ]:
# ── Part 6 · dust masses: simulation prediction vs CIGALE ────────────────────
rng = np.random.default_rng(4)
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True)
summary = []
for ax, (tid, T) in zip(axes, TARGETS.items()):
    A = ANALOGUES[tid]
    for x0, cls in ((0.0, 'no_AGN'), (1.0, 'AGN')):
        G = A[np.asarray(A['AGN_CLASS']) == cls]
        y = np.asarray(G['LOG_MDUST'], float)
        y = y[np.isfinite(y)]
        x = x0 + rng.uniform(-0.16, 0.16, y.size)
        ax.plot(x, y, 'o', ms=4.5, color=COL[cls], alpha=0.45, mec='none')
        if y.size:
            med, lo, hi = np.median(y), np.percentile(y, 16), np.percentile(y, 84)
            ax.hlines(med, x0 - 0.24, x0 + 0.24, color=COL[cls], lw=2.5)
            ax.vlines(x0 - 0.24, lo, hi, color=COL[cls], lw=1.2)
            summary.append(dict(target=tid, group=cls, N=int(y.size), med_logMd=round(float(med), 2),
                                p16=round(float(lo), 2), p84=round(float(hi), 2)))
    # observed CIGALE value: 719 is effectively an upper limit (err > value)
    if tid == '719':
        ax.errorbar(0.5, T['logmdust'], yerr=[[0.0], [0.0]], uplims=True, fmt='s', ms=9,
                    color=COL['obs'], capsize=4, label='CIGALE (~upper limit)')
    else:
        ax.errorbar(0.5, T['logmdust'], yerr=T['logmdust_err'], fmt='s', ms=9,
                    color=COL['obs'], capsize=4, label='CIGALE')
    ax.set(xticks=[0, 0.5, 1], xticklabels=['non-AGN\nhosts', 'observed', 'AGN\nhosts'],
           xlim=(-0.5, 1.5), title=f'nirspec7-{tid.zfill(6)}')
    ax.grid(alpha=0.2, axis='y', lw=0.5)
axes[0].set_ylabel(r'$\log\,M_{\rm dust}\,[M_\odot]$')
fig.tight_layout()
for ext in ('png', 'pdf'):
    fig.savefig(os.path.join(OUTDIR, f'dust_mass_analogues.{ext}'), dpi=200, bbox_inches='tight')
plt.show()

SUM = Table(summary)
SUM.write(os.path.join(OUTDIR, 'dust_mass_summary.fits'), overwrite=True)
SUM.pprint(max_width=-1)
print(f"\nobserved: 719 log Md ~ {TARGETS['719']['logmdust']} (upper-limit-like), "
      f"2962 log Md = {TARGETS['2962']['logmdust']} +/- {TARGETS['2962']['logmdust_err']}")


In [ ]:
# ── Part 6b · A_V: powderday (attenuated / unattenuated) vs CIGALE ───────────
# A_V measured directly from the RT: -2.5 log10(L_att / L_unatt) at rest-frame V,
# pairing the two AGN-on Chabrier arms (dusty_simdust_chab_agn vs nodust_chab_agn) —
# the only physically matched pair available (there is no nodust AGN-off arm).
# Caveat: for AGN-class hosts this is the TOTAL (stellar+AGN) effective attenuation
# at V, while CIGALE's bayes.attenuation.generic.bessell.V is the stellar-continuum
# attenuation (the SKIRTOR AGN component carries its own E(B-V)).

# observed side: CIGALE bayesian V-band attenuation from the chab run
RES = Table.read(os.path.join(OBS_DIR, 'results.fits'))
_res_id = np.asarray([str(i).strip() for i in RES['id']])
CIG_AV = {}
for tid in TARGETS:
    r = RES[_res_id == tid][0]
    CIG_AV[tid] = (float(r['bayes.attenuation.generic.bessell.V']),
                   float(r['bayes.attenuation.generic.bessell.V_err']))

VWIN = np.linspace(0.54, 0.56, 25)   # rest micron, ~Bessell V core (curves are smooth there)

def av_powderday(snap, gid):
    """A_V = -2.5 log10(F_att/F_unatt) at rest V; None if either RT is unfinished."""
    for root in (AGN_SED_ROOT, NODUST_SED_ROOT):
        if root is None or not sed_is_complete(rtout_path(snap, gid, root)):
            return None
    w_on,  l_on  = read_powderday_sed(snap, gid, AGN_SED_ROOT)
    w_off, l_off = read_powderday_sed(snap, gid, NODUST_SED_ROOT)
    f_on  = np.median(np.interp(VWIN, w_on,  l_on))
    f_off = np.median(np.interp(VWIN, w_off, l_off))
    if not (f_on > 0 and f_off > 0):
        return None
    return -2.5 * np.log10(f_on / f_off)

rng = np.random.default_rng(4)
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True)
summary = []
for ax, (tid, T) in zip(axes, TARGETS.items()):
    A = ANALOGUES[tid]
    n_skip = 0
    for x0, cls in ((0.0, 'no_AGN'), (1.0, 'AGN')):
        G = A[np.asarray(A['AGN_CLASS']) == cls]
        vals = [av_powderday(r['SNAPSHOT'], r['GROUPID_SNAPSHOT']) for r in G]
        n_skip += sum(v is None for v in vals)
        y = np.asarray([v for v in vals if v is not None], float)
        x = x0 + rng.uniform(-0.16, 0.16, y.size)
        ax.plot(x, y, 'o', ms=4.5, color=COL[cls], alpha=0.45, mec='none')
        if y.size:
            med, lo, hi = np.median(y), np.percentile(y, 16), np.percentile(y, 84)
            ax.hlines(med, x0 - 0.24, x0 + 0.24, color=COL[cls], lw=2.5)
            ax.vlines(x0 - 0.24, lo, hi, color=COL[cls], lw=1.2)
            summary.append(dict(target=tid, group=cls, N=int(y.size), med_av=round(float(med), 3),
                                p16=round(float(lo), 3), p84=round(float(hi), 3)))
    av, av_err = CIG_AV[tid]
    ax.errorbar(0.5, av, yerr=av_err, fmt='s', ms=9, color=COL['obs'], capsize=4, label='CIGALE')
    ax.set(xticks=[0, 0.5, 1], xticklabels=['non-AGN\nhosts', 'observed', 'AGN\nhosts'],
           xlim=(-0.5, 1.5), title=f'nirspec7-{tid.zfill(6)}')
    ax.grid(alpha=0.2, axis='y', lw=0.5)
    print(f"{tid}: {n_skip} analogue(s) skipped (unfinished attenuated or nodust RT)")
axes[0].set_ylabel(r'$A_V$ [mag]')
fig.tight_layout()
for ext in ('png', 'pdf'):
    fig.savefig(os.path.join(OUTDIR, f'av_analogues.{ext}'), dpi=200, bbox_inches='tight')
plt.show()

SUM = Table(summary)
SUM.write(os.path.join(OUTDIR, 'av_summary.fits'), overwrite=True)
SUM.pprint(max_width=-1)
print(f"\nobserved (CIGALE bayes bessell.V): "
      f"719 A_V = {CIG_AV['719'][0]:.3f} +/- {CIG_AV['719'][1]:.3f}, "
      f"2962 A_V = {CIG_AV['2962'][0]:.3f} +/- {CIG_AV['2962'][1]:.3f}")


In [ ]:
# ── Part 6c · A_V vs dust mass: does attenuation track dust content? ─────────
# Per-galaxy scatter over both analogue samples (needs Part 6b in memory: av_powderday,
# CIG_AV). If the dusty tail of the 2962 analogues climbs onto the observed points,
# SIMBA reproduces the observed attenuation *when* it matches the dust mass, i.e. the
# discrepancy is dust content, not attenuation physics/geometry.
pts = {}   # (tid, cls) -> (logMd, A_V) arrays, RT-complete galaxies only
for tid in TARGETS:
    A = ANALOGUES[tid]
    for cls in ('no_AGN', 'AGN'):
        G = A[np.asarray(A['AGN_CLASS']) == cls]
        md, av = [], []
        for r in G:
            a = av_powderday(r['SNAPSHOT'], r['GROUPID_SNAPSHOT'])
            m = float(r['LOG_MDUST'])
            if a is None or not np.isfinite(m):
                continue
            md.append(m); av.append(a)
        pts[(tid, cls)] = (np.asarray(md, float), np.asarray(av, float))

MARK = {'719': 'o', '2962': '^'}
fig, ax = plt.subplots(figsize=(7.2, 5.2))
n_drop = 0
for (tid, cls), (md, av) in pts.items():
    pos = av > 0                      # log y-axis: A_V <= 0 (RT noise around zero) can't show
    n_drop += int((~pos).sum())
    ax.plot(md[pos], av[pos], MARK[tid], ms=5, color=COL[cls], alpha=0.55, mec='none',
            label=f"{tid} {cls.replace('_', '-')} (N={int(pos.sum())})")
for tid, T in TARGETS.items():
    av, av_err = CIG_AV[tid]
    if tid == '719':   # CIGALE dust mass is ~an upper limit
        ax.errorbar(T['logmdust'], av, yerr=av_err, xerr=[[0.35], [0.0]], xuplims=True,
                    fmt='s', ms=9, color=COL['obs'], capsize=4, zorder=5, label='CIGALE')
    else:
        ax.errorbar(T['logmdust'], av, yerr=av_err, xerr=T['logmdust_err'],
                    fmt='s', ms=9, color=COL['obs'], capsize=4, zorder=5)
    ax.annotate(f'nirspec7-{tid}', (T['logmdust'], av), textcoords='offset points',
                xytext=(8, 6), fontsize=9, color=COL['obs'])
xg = np.array([3.4, 7.6])             # slope-1 guide through the 2962 CIGALE point:
ax.plot(xg, 10**(xg - TARGETS['2962']['logmdust']) * CIG_AV['2962'][0],   # A_V ∝ M_d at
        ls=':', color='0.5', lw=1, label=r'$A_V \propto M_{\rm dust}$ (through 2962)')
ax.set(xlabel=r'$\log\,M_{\rm dust}\,[M_\odot]$', ylabel=r'$A_V$ [mag]', yscale='log')
ax.grid(alpha=0.2, lw=0.5)
ax.legend(fontsize=8, loc='upper left', frameon=False)
fig.tight_layout()
for ext in ('png', 'pdf'):
    fig.savefig(os.path.join(OUTDIR, f'av_vs_mdust_analogues.{ext}'), dpi=200, bbox_inches='tight')
plt.show()
if n_drop:
    print(f'{n_drop} galaxy(ies) with A_V <= 0 (RT noise) not shown on the log axis')

# rank correlation per target (both AGN classes together, A_V <= 0 kept: Spearman is rank-based)
def _spearman(x, y):
    rx = np.argsort(np.argsort(x)).astype(float)
    ry = np.argsort(np.argsort(y)).astype(float)
    rx -= rx.mean(); ry -= ry.mean()
    return float((rx * ry).sum() / np.sqrt((rx**2).sum() * (ry**2).sum()))
for tid in TARGETS:
    md = np.concatenate([pts[(tid, c)][0] for c in ('no_AGN', 'AGN')])
    av = np.concatenate([pts[(tid, c)][1] for c in ('no_AGN', 'AGN')])
    print(f'{tid}: Spearman rho(A_V, log Md) = {_spearman(md, av):+.2f}  (N={md.size})')


In [ ]:
"""Part 6g · single-galaxy look at the AGN-on powderday SED.

Self-contained: needs nothing from the notebook's earlier cells, so it can be `%run` from
anywhere or pasted in as two cells (split at the `PART 6g-b` banner).

    %run part6g_single_galaxy.py

Written 2026-07-31 for `analogues_specphot_cis100_rt_agn.ipynb`.

WHY THIS EXISTS
---------------
A `.rtout.sed` on disk is NOT proof the RT finished. hyperion creates the output file when it
*starts* and appends the grid arrays after each Lucy iteration; the `Peeled` groups that hold the
SEDs are written, and the file closed, only at the very end. A job still in its raytracing pass
therefore leaves a large, complete-looking `.rtout.sed` that h5py refuses with

    OSError: Unable to open file (bad object header version number)

`sed_is_complete()` below is the test. `os.path.exists` is not — and Parts 6a/6b of the notebook
currently use `os.path.exists`, which is exactly why a still-running job looked finished on
2026-07-31 (snap 091 / gal 1747: rtout mtime 02:26, but the LOG still live at 08:05, ~6 h into
`[main] starting raytracing iteration`).
"""

import collections
import glob
import os
import time

import numpy as np
import h5py
from astropy.table import Table
from astropy import constants as const
import astropy.units as u

# ── configuration (mirrors Part 0 of the notebook) ──────────────────────────
HOME = '/mnt/home/glorenzon/analize_simba_cgm'
OUTDIR = os.path.join(HOME, 'output', 'cis100', 'analogues_specphot')
SED_OUT = os.path.join(HOME, 'output', 'cis100', 'sed_analogues')

TAG_AGN = 'dusty_simdust_chab_agn'      # AGN-on, corrected (Chabrier) list
TAG_OFF_TOPUP = 'dusty_simdust_chab_topup'    # AGN-off for galaxies not already run
TAG_OFF_DONE = 'dusty_simdust'               # finished AGN-off (read only)

TARGETS = {'719': dict(snap=91, z_snap=1.4584),
           '2962': dict(snap=96, z_snap=1.2780)}
SNAPS_ALL = sorted({int(T['snap']) for T in TARGETS.values()})


def _rtout_p(tag, snap, gid):
    """Path of the RT output for one galaxy in one run tree (may not exist / be incomplete)."""
    snap, gid = int(snap), int(gid)
    return os.path.join(SED_OUT, tag, 'powderday_sed_out', f'snap_{snap:03d}',
                        f'gal_{gid}', f'snap{snap:03d}.galaxy{gid:06d}.rtout.sed')


def sed_is_complete(path):
    """True only if the RT actually FINISHED — existence of the file is not enough.

    See the module docstring: hyperion writes the `Peeled` SED groups last. Use this everywhere
    the notebook currently uses `os.path.exists` on an .rtout.sed.
    """
    if not (path and os.path.exists(path)):
        return False
    try:
        with h5py.File(path, 'r') as f:
            g = f.get('Peeled')
            return bool(g) and any('seds' in g[k] for k in g.keys())
    except OSError:
        return False


def _as_str(col):
    """FITS string columns come back as bytes (|S6); `== 'AGN'` on them matches nothing."""
    return np.array([x.decode() if isinstance(x, (bytes, np.bytes_)) else str(x) for x in col])


def _source_types(rtout):
    """Multiset of hyperion source types. Stellar sources are `point_collection`; an AGN is
    `point` (`m.add_point_source`). Attributes only — do NOT walk the datasets, it is slow."""
    with h5py.File(rtout, 'r') as f:
        src = f['Input/Sources']
        out = []
        for k in src.keys():
            t = src[k].attrs.get('type', b'?')
            out.append(t.decode() if isinstance(t, bytes) else str(t))
        return collections.Counter(out)


# ═══════════════════════════════════════════════════════════════════════════
# PART 6g-a · which galaxies are REALLY finished, and where the rest are
# ═══════════════════════════════════════════════════════════════════════════

def log_phase(gdir):
    """Last milestone the powderday/hyperion LOG reached, and how stale it is (minutes).

    A live LOG means the job is alive; a stale LOG on an incomplete SED means it died.
    """
    logs = glob.glob(os.path.join(gdir, '*.LOG'))
    if not logs:
        return 'no LOG (never started)', float('nan')
    txt = open(logs[0], errors='replace').read()
    if 'Total CPU time elapsed' in txt or 'Ending on' in txt:
        phase = 'hyperion finished'
    else:
        phase = 'python front end'
        for key in ('starting raytracing iteration',      # most advanced first
                    'starting final iteration',
                    'starting Lucy iteration',
                    'Adding Black Holes to Source List'):
            if key in txt:
                phase = key
                break
    return phase, (time.time() - os.path.getmtime(logs[0])) / 60.0


def scan_trees(tags=(TAG_AGN, TAG_OFF_TOPUP), show=20):
    """Per-tree census: complete / still-being-written / started / never started."""
    complete = {}
    for tag in tags:
        rows = []
        for snap in SNAPS_ALL:
            jdir = os.path.join(SED_OUT, tag, 'powderday_sed_out', f'snap_{snap:03d}')
            for gdir in sorted(glob.glob(os.path.join(jdir, 'gal_*'))):
                gid = int(os.path.basename(gdir).split('_')[1])
                p = _rtout_p(tag, snap, gid)
                ph, age = log_phase(gdir)
                rows.append((snap, gid, os.path.exists(p), sed_is_complete(p), ph, age))

        done = [r for r in rows if r[3]]
        writing = [r for r in rows if r[2] and not r[3]]
        started = [r for r in rows if not r[2] and 'never started' not in r[4]]
        print(f'\n=== {tag} ===')
        print(f'  staged                                    : {len(rows)}')
        print(f'  COMPLETE (SEDs readable)                  : {len(done)}')
        print(f'  .rtout.sed present but still being written : {len(writing)}'
              '   <- NOT finished')
        print(f'  started, no .rtout.sed yet                : {len(started)}')
        for r in (done + writing)[:show]:
            flag = 'COMPLETE' if r[3] else 'writing '
            stale = '' if r[5] != r[5] else (' (STALE — job dead?)' if r[5] > 90 else '')
            print(f'    snap {r[0]:03d} gal {r[1]:<6d} {flag}  {r[4]:<34} '
                  f'LOG {r[5]:6.1f} min old{stale}')
        complete[tag] = [(r[0], r[1]) for r in done]
    print('\nAGN-off tree (already finished, read-only):', TAG_OFF_DONE)
    return complete


# ═══════════════════════════════════════════════════════════════════════════
# PART 6g-b · one galaxy: AGN-on vs AGN-off, and what the torus added
# ═══════════════════════════════════════════════════════════════════════════
# Same galaxy, same dust, same grid, same photon budget — only BH_SED differs — so
# (AGN-on − AGN-off) IS the torus, with no modelling in between.

def _selection():
    """rt_selection_chab_*.fits keyed by target, plus the AGN-off provenance map."""
    sel = {}
    for tid in TARGETS:
        A = Table.read(os.path.join(OUTDIR, f'rt_selection_chab_{tid}.fits'))
        A['AGN_CLASS'] = _as_str(A['AGN_CLASS'])
        sel[tid] = A
    om = Table.read(os.path.join(OUTDIR, 'agnoff_source_map_chab.fits'))
    off_tag = dict(zip(zip(np.asarray(om['SNAPSHOT'], int),
                           np.asarray(om['GROUPID_SNAPSHOT'], int)),
                       _as_str(om['AGNOFF_TAG'])))
    return sel, off_tag


def _sed1(path):
    """(wav_rest [um], nu*Lnu [erg/s]) — largest aperture, mean over the 4 sightlines.

    THETA=[0,45,90,135] / PHI=[0,90,180,270] are 4 genuine sightlines, but for these dust-poor
    quenched galaxies they agree to <0.5%, so the mean is safe.
    """
    from hyperion.model import ModelOutput
    s = ModelOutput(path).get_sed(inclination='all', aperture=-1)
    w = np.asarray(s.wav, float)
    v = np.atleast_2d(np.asarray(s.val, float)).mean(axis=0)
    o = np.argsort(w)
    return w[o], v[o]


def _Lbol_sed(w, v):
    """Integrate a nu*Lnu curve: int Lnu dnu = int (nu*Lnu)/nu dnu   [erg/s]."""
    nu = (const.c / (w * u.micron)).to(u.Hz).value
    o = np.argsort(nu)
    return float(np.trapz(v[o] / nu[o], nu[o]))


def single_galaxy(gal=None, complete=None, save=True):
    """Plot AGN-on vs AGN-off for one galaxy. `gal=(snap, gid)`, or None for the first
    COMPLETE galaxy in the AGN tree."""
    import matplotlib.pyplot as plt

    if gal is None:
        if complete is None:
            complete = scan_trees()
        if not complete.get(TAG_AGN):
            raise RuntimeError(
                'No galaxy in the AGN tree has a complete .rtout.sed yet — the RT is still in '
                'its raytracing pass (the census above shows where). Nothing can be plotted '
                'until at least one galaxy is COMPLETE.')
        gal = complete[TAG_AGN][0]
    snap1, gid1 = int(gal[0]), int(gal[1])

    sel, off_tag = _selection()
    row = tid1 = None
    for tid, A in sel.items():
        m = ((np.asarray(A['SNAPSHOT'], int) == snap1) &
             (np.asarray(A['GROUPID_SNAPSHOT'], int) == gid1))
        if m.any():
            row, tid1 = A[m][0], tid
            break
    if row is None:
        raise RuntimeError(f'snap {snap1} / gal {gid1} is not in any rt_selection_chab_*.fits')
    z1 = float(row['REDSHIFT'])

    p_on = _rtout_p(TAG_AGN, snap1, gid1)
    _tag_off = off_tag.get((snap1, gid1))
    p_off = None if _tag_off is None else _rtout_p(_tag_off, snap1, gid1)
    if not sed_is_complete(p_on):
        raise RuntimeError(f'the AGN-on SED of {snap1}/{gid1} is not finished ({p_on})')
    if not sed_is_complete(p_off):
        raise RuntimeError(f'the AGN-off counterpart of {snap1}/{gid1} is not finished '
                           f'({p_off}) — nothing to difference against.')

    w1, l_on = _sed1(p_on)
    w2, l_off = _sed1(p_off)
    if not np.allclose(w1, w2):
        raise RuntimeError('the two runs used different wavelength grids — cannot difference')

    # the intrinsic AGN template powderday injected (dump_AGN_SEDs)
    w_bh = l_bh = None
    l_inj = np.nan
    npz = glob.glob(os.path.join(os.path.dirname(p_on), 'bh_sed*.npz'))
    if npz:
        with np.load(npz[0]) as z:
            nu_bh = np.asarray(z['nu'], float)                                # Hz
            fnu_bh = np.atleast_2d(np.asarray(z['fnu'], float)).sum(axis=0)   # over holes
            l_inj = float(np.sum(z['luminosity']))                            # erg/s
        o = np.argsort(nu_bh)
        integ = float(np.trapz(fnu_bh[o], nu_bh[o]))
        # fnu is in powderday's internal units; renormalise to the L_bol it reported so the
        # template shares the erg/s axis. Not expected to need a factor of 1 — the stored nu
        # grid (1e12–3e15 Hz) is truncated relative to the full Hopkins template.
        wv = (const.c / (nu_bh * u.Hz)).to(u.micron).value
        p = np.argsort(wv)
        w_bh = wv[p]
        l_bh = (fnu_bh * nu_bh)[p] * (l_inj / integ if integ > 0 else 1.0)

    b_on, b_off = _Lbol_sed(w1, l_on), _Lbol_sed(w1, l_off)
    print(f'galaxy           : snap {snap1:03d} / gal {gid1}  '
          f'(analogue of nirspec7-{tid1.zfill(6)}, z_snap={z1:.4f})')
    print(f'  logM*, age     : {float(row["LOG_MSTAR"]):.3f}, {float(row["AGE_STAR"]):.2f} Gyr   '
          f'sSFR={float(row["SSFR"]):.2e} /yr')
    print(f'  AGN class      : {row["AGN_CLASS"]}   f_Edd={float(row["FEDD"]):.4f}   '
          f'L_bol(caesar)={float(row["LBOL"]):.3e} erg/s')
    print(f'  M_dust         : {float(row["MDUST"]):.3e} Msun   '
          f'(N_gas={int(row["NGAS"])}, N_star={int(row["NSTAR"])})')
    print(f'  sources on     : {dict(_source_types(p_on))}   '
          "<- an AGN is a 'point'; stars are 'point_collection'")
    print(f'  sources off    : {dict(_source_types(p_off))}')
    print(f'  L_AGN injected : {l_inj:.4e} erg/s'
          + (f'   -> injected / caesar L_bol = {l_inj / float(row["LBOL"]):.3f}'
             if float(row['LBOL']) > 0 else ''))
    print(f'  L_bol(SED) on  : {b_on:.4e} erg/s')
    print(f'  L_bol(SED) off : {b_off:.4e} erg/s')
    print(f'  on - off       : {b_on - b_off:+.4e} erg/s'
          + (f'  = {(b_on - b_off) / l_inj:.3f} x L_AGN injected   '
             '(dust reprocesses but conserves energy, so ~1 unless the template runs off '
             'the SED grid)' if np.isfinite(l_inj) and l_inj > 0 else ''))

    probe = (0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 60.0, 200.0)
    ratio = l_on / l_off
    print('\n  rest wav [um] :' + ''.join(f'{p:9.1f}' for p in probe))
    print('  on / off      :' + ''.join(
        f'{ratio[int(np.argmin(np.abs(w1 - p)))]:9.3f}' for p in probe))

    # ── figure ─────────────────────────────────────────────────────────────
    # Okabe-Ito, as elsewhere in these notebooks: blue = AGN-off, orange = AGN-on.
    c_off, c_on, c_bh, c_dif = '#0072B2', '#E69F00', '#7F7F7F', '#009E73'
    plt.rcParams.update({'font.size': 12})
    fig, (ax, axr) = plt.subplots(2, 1, figsize=(9.2, 8.0), sharex=True,
                                  gridspec_kw=dict(height_ratios=[2.6, 1.0], hspace=0.06))

    if l_bh is not None:
        ax.plot(w_bh, l_bh, color=c_bh, lw=1.4, ls=':',
                label='injected AGN template (Hopkins)')
    ax.plot(w1, l_off, color=c_off, lw=2.0, label='AGN-off')
    ax.plot(w1, l_on, color=c_on, lw=2.0, label='AGN-on')
    d = l_on - l_off
    ax.plot(w1, np.where(d > 0, d, np.nan), color=c_dif, lw=1.4, ls='--',
            label='AGN-on $-$ AGN-off')

    fin = np.isfinite(l_on) & (l_on > 0)
    ax.set(xscale='log', yscale='log', xlim=(0.05, 1000),
           ylim=(np.nanmax(l_on[fin]) * 1e-6, np.nanmax(l_on[fin]) * 3),
           ylabel=r'$\nu L_\nu$  [erg s$^{-1}$]  (rest frame)')
    ax.set_title(f'snap {snap1:03d} / gal {gid1} — analogue of nirspec7-{tid1.zfill(6)}, '
                 f'z={z1:.3f}, log$M_\\star$={float(row["LOG_MSTAR"]):.2f}, '
                 f'{row["AGN_CLASS"]}', fontsize=12)
    ax.grid(alpha=0.18, which='both', lw=0.5)
    ax.legend(fontsize=10, frameon=False, loc='lower center', ncol=2)

    # observed-frame wavelength at this galaxy's z — the same axis, relabelled (not a 2nd scale)
    axt = ax.secondary_xaxis('top', functions=(lambda x: x * (1 + z1),
                                               lambda x: x / (1 + z1)))
    axt.set_xlabel(r'observed-frame wavelength [$\mu$m]  (z = %.3f)' % z1, fontsize=11)

    axr.plot(w1, ratio, color=c_on, lw=2.0)
    axr.axhline(1.0, color='k', lw=0.8, ls='--')
    axr.set(xscale='log', yscale='log', xlim=(0.05, 1000),
            xlabel=r'rest-frame wavelength [$\mu$m]',
            ylabel=r'$\nu L_\nu$ on / off')
    axr.grid(alpha=0.18, which='both', lw=0.5)
    fig.tight_layout()

    if save:
        stem = os.path.join(OUTDIR, f'agn_single_snap{snap1:03d}_gal{gid1}')
        for ext in ('png', 'pdf'):
            fig.savefig(f'{stem}.{ext}', dpi=200, bbox_inches='tight')
        print('\n->', stem + '.png')
    plt.show()
    return dict(snap=snap1, gid=gid1, target=tid1, z=z1, wav=w1, nuLnu_on=l_on,
                nuLnu_off=l_off, ratio=ratio, L_inj=l_inj, L_on=b_on, L_off=b_off)


if __name__ == '__main__':
    COMPLETE = scan_trees()
    if COMPLETE.get(TAG_AGN):
        RES = single_galaxy(complete=COMPLETE)
    else:
        print('\nNothing COMPLETE in the AGN tree yet — re-run this once the RT has finished a '
              'galaxy, then call  single_galaxy()  (or  single_galaxy((91, 1747))  to force one).')

## Part 7 · status and caveats

**The redshift gap is closed.** The earlier version of this notebook read
`sed_dust_comparison/dusty_simdust`, whose RT only ever completed at snap >= 100 (z <= 1.145), and
matched with loose cosmic-time windows - so every analogue sat below both target redshifts. The run
staged by `analogues_specphot_cis100_rt.ipynb` and finished on 2026-07-28/29 closed that gap;
the notebook now reads the corrected Chabrier-window set staged by
`analogues_specphot_cis100_rt_agn.ipynb`:

| target | z_obs | snap | z_snap | \|dz\| | N (AGN / non-AGN) |
|---|---|---|---|---|---|
| 000719 | 1.463 | 091 | 1.4584 | 0.005 | 60 (20 / 40) |
| 002962 | 1.266 | 096 | 1.2780 | 0.012 | 56 (16 / 40) |

**Caveats**

- **No AGN emission in this run.** `parameters_master.py` has `BH_SED = False`, so the simulated
  SEDs contain no torus even for AGN hosts; the split tests the *host* (selected on `FEDD`/`LBOL`).
  For 002962 compare the AGN-host band to the **dashed galaxy-only** CIGALE curve, never the total.
  The AGN-on twin is staged by `analogues_specphot_cis100_rt_agn.ipynb`
  (`sed_analogues/dusty_simdust_chab_agn`, `BH_SED=True`, Hopkins template, `BH_var=False`);
  `PartType5` was re-extracted into the cutouts on 2026-07-30 and verified against caesar `nbh`.
  `AGN_SED_ROOT` points at that tree and Part 5 overlays every galaxy whose RT is finished.

- **Sightlines.** `THETA = [0, 45, 90, 135]`, `PHI = [0, 90, 180, 270]` are 4 genuine viewing
  angles, but for these dust-poor quenched galaxies the SEDs agree to < 0.5%, so the stacks carry
  essentially no inclination scatter. Total aperture (largest of the 5-rung 1-100 kpc ladder),
  `kmh94` dust, live SIMBA dust grid (`dust_grid_type='manual'`), 100 kpc zoom box.
- **Two AGN definitions.** `IS_AGN` (f_Edd >= 0.02 AND L_bol >= 1e43) is the primary split;
  `IS_AGN_LBOL` (L_bol >= 1e43 only) is carried alongside because the f_Edd gate is what removes
  the entire AGN class in the cis25 box. Part 4 prints the cross-tab; `SPLIT_BY` in Part 5 selects.
- 719's Bagpipes-preferred *young/low-mass branch* (log M* = 9.98, 0.9 Gyr) lies below the staged
  mass window and cannot be matched by construction; this comparison tests the CIGALE/pPXF branch.
- Rest-frame conversion uses nu_obs*F_nu*4*pi*d_L^2 = nu_rest*L_nu,rest with Planck13, consistent
  with `makesed`.

**Convergence.** Every RT output carries `converged = 'no'` (stopped at the 7-iteration cap set by `set_n_initial_iterations(7)` in powderday's `front_end_tools.py`). Checked against the run logs: the convergence metric is **flat** at 1.07-1.24 vs the 1.01 threshold with `Difference from previous iteration: 1.00` (zero improvement) from iteration 2 onward, and each log has 45 `specific_energy below minimum allowed` warnings. This is a Monte-Carlo **photon** noise floor in the sparsest outskirt cells of the +/-100 kpc box, not an iteration shortfall - raising the cap would change nothing. Since the criterion is the worst 1% of cells (negligible dust mass, negligible SED contribution), it is largely cosmetic. The lever, if ever needed, is `n_photons_initial` (currently 1e7), not more iterations.
